# Step 9: Matrix Product States by Hand via SVD Truncation

In notebook 08 we measured the entanglement entropy and saw that gapped ground states satisfy the area law: the Schmidt spectrum decays exponentially and only a small number of singular values are significant. The MPS representation makes this structure explicit.

In this notebook we take the TFIM ground state from exact diagonalisation and decompose it into an MPS by performing a sequence of singular value decompositions — one per bond. This is the operation that runs inside DMRG every time a site tensor is updated. Understanding it here, by building it from scratch, is the bridge to notebook 10.

**What you will do:**
- Understand the MPS structure: matrices contracted over virtual indices
- Build an exact MPS from any ED wavefunction via sequential SVD
- Verify reconstruction to machine precision
- Truncate the bond dimension and measure the resulting error
- Compare the required bond dimension in gapped vs critical phases
- Compute expectation values directly from MPS tensors

**Prerequisites:** Notebooks 07–08 (TFIM ground state, Schmidt decomposition, entanglement).

## 9.1  What Is a Matrix Product State?

A many-body wavefunction has $d^N$ coefficients — exponential in system size. An MPS writes the same state as a product of small matrices:

$$|\psi\rangle = \sum_{s_0, \ldots, s_{N-1}} A_0^{s_0}\, A_1^{s_1}\, \cdots\, A_{N-1}^{s_{N-1}}\, |s_0 s_1 \cdots s_{N-1}\rangle$$

Each tensor $A_i$ has three indices:
- **Left virtual index** $\alpha$ of dimension $\chi_{i-1}$ (the **bond dimension**)
- **Physical index** $s_i \in \{0, 1\}$ selecting the local spin state
- **Right virtual index** $\beta$ of dimension $\chi_i$

For a fixed physical configuration $(s_0, s_1, \ldots, s_{N-1})$, selecting one $d \times d$ matrix slice $A_i^{s_i}$ at each site and multiplying them gives the coefficient — a scalar because the boundary dimensions $\chi_{-1} = \chi_{N-1} = 1$.

The total parameter count is $O(N \chi^2 d)$ — polynomial in both $N$ and $\chi$. The exponential compression relative to the full wavefunction is only justified when $\chi$ can be kept small, which is precisely what the area law guarantees for gapped states.

**Every state is an exact MPS** — the bond dimension just grows. For a product state $\chi = 1$ exactly. For a general entangled state the exact $\chi$ can reach $d^{N/2}$, reproducing the exponential. The approximation comes from **truncating** to a fixed $\chi$, discarding the smallest singular values at each bond.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

# ── TFIM Hamiltonian (from notebook 06) ──────────────────────────────────────
def build_tfim(N, J=1.0, Gamma=1.0, pbc=False):
    dim    = 2 ** N
    states = np.arange(dim)
    rows, cols, vals = [], [], []
    n_bonds = N if pbc else N - 1
    for i in range(n_bonds):
        j    = (i + 1) % N
        bi   = (states >> i) & 1
        bj   = (states >> j) & 1
        sz_i = 2 * bi.astype(float) - 1
        sz_j = 2 * bj.astype(float) - 1
        rows.extend(states.tolist())
        cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())
    for i in range(N):
        s_flip = states ^ (1 << i)
        rows.extend(states.tolist())
        cols.extend(s_flip.tolist())
        vals.extend([-Gamma] * dim)
    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)


# ── Encoding conversion: LSB (our ED convention) <-> MSB (MPS convention) ────
# Our ED encoding: psi[s] with bit i = (s>>i)&1 = spin at site i (LSB = site 0)
# MPS convention below uses C-order: site 0 occupies the most significant position,
# giving psi_c[s0*d^{N-1} + s1*d^{N-2} + ... + s_{N-1}].
# Conversion: reshape the LSB vector as a d^N tensor in F-order and flatten in C-order.

def lsb_to_msb(psi, N, d=2):
    """Convert LSB-encoded state (ED convention) to MSB-encoded (MPS convention)."""
    return psi.reshape([d] * N, order='F').reshape(-1, order='C')

def msb_to_lsb(psi_c, N, d=2):
    """Convert MSB-encoded state (MPS convention) back to LSB-encoded (ED convention)."""
    return psi_c.reshape([d] * N, order='C').reshape(-1, order='F')


# Sanity check: round-trip on a random state
N_test = 4
psi_rand = np.random.randn(2 ** N_test)
psi_rand /= np.linalg.norm(psi_rand)
error = np.max(np.abs(msb_to_lsb(lsb_to_msb(psi_rand, N_test), N_test) - psi_rand))
print(f"Encoding round-trip error: {error:.2e}  (expected ~0)")

## 9.2  Constructing an Exact MPS by Sequential SVD

Given the coefficient tensor $c_{s_0 s_1 \cdots s_{N-1}}$, we convert to MPS form by a left-to-right sweep of SVDs:

**Step $i$:** Reshape the current matrix (of shape $\chi_{i-1} d \times d^{N-i-1}$) and SVD:
$$M_i = U_i\, S_i\, V_i^\dagger$$
The columns of $U_i$, reshaped to $(\chi_{i-1}, d, \chi_i)$, become the MPS tensor $A_i$. The singular values $S_i$ encode the Schmidt spectrum at bond $i$. The remainder $S_i V_i^\dagger$ is passed to the next step.

The result is a **left-canonical MPS**: each tensor satisfies $\sum_{s_i} (A_i^{s_i})^\dagger A_i^{s_i} = I$, which is automatically guaranteed by the orthogonality of the SVD left singular vectors.

In [ ]:
def state_to_mps(psi_c, N, d=2, chi_max=None, sv_tol=1e-14):
    """
    Convert MSB-encoded state vector to left-canonical MPS by sequential SVD.

    psi_c[s0 * d^{N-1} + ... + s_{N-1}]: coefficient tensor in C-order.
    chi_max: maximum bond dimension (None = keep all singular values above sv_tol).
    sv_tol: threshold below which singular values are treated as zero.

    Returns: list of N tensors, each of shape (chi_left, d, chi_right).
    Also returns: list of N-1 singular value arrays (one per bond).
    """
    tensors   = []
    sv_bonds  = []      # singular values at each bond
    chi_left  = 1
    remaining = psi_c.astype(float).copy()

    for i in range(N - 1):
        d_right = d ** (N - i - 1)
        mat     = remaining.reshape(chi_left * d, d_right)

        U, s, Vh = np.linalg.svd(mat, full_matrices=False)

        # Determine how many singular values to keep
        n_keep = int(np.sum(s > sv_tol))
        if chi_max is not None:
            n_keep = min(n_keep, chi_max)
        n_keep = max(n_keep, 1)   # always keep at least one

        U  = U[:, :n_keep]
        s  = s[:n_keep]
        Vh = Vh[:n_keep, :]

        # MPS tensor: shape (chi_left, d, chi_right)
        tensors.append(U.reshape(chi_left, d, n_keep))
        sv_bonds.append(s)

        # Pass singular values into the right block
        remaining = s[:, None] * Vh
        chi_left  = n_keep

    # Last site: remaining has shape (chi_{N-1}, d)
    tensors.append(remaining.reshape(chi_left, d, 1))
    return tensors, sv_bonds


def mps_to_state(tensors, N, d=2):
    """
    Contract MPS tensors to recover the MSB-encoded state vector.
    """
    state = tensors[0][0, :, :]    # shape (d, chi_1)
    for i in range(1, N):
        A         = tensors[i]     # shape (chi_i, d, chi_{i+1})
        state     = np.einsum('ia,ajb->ijb', state, A)   # (d^i, d, chi_{i+1})
        chi_next  = A.shape[2]
        state     = state.reshape(-1, chi_next)
    return state[:, 0]


# ── Test on small TFIM ground state ─────────────────────────────────────────
J     = 1.0
N_ex  = 12

for label, Gamma in [('Ordered (Γ/J=0.3)', 0.3), ('Critical (Γ=J)', 1.0), ('Disordered (Γ/J=2)', 2.0)]:
    H       = build_tfim(N_ex, J=J, Gamma=Gamma, pbc=False)
    _, evec = eigsh(H, k=1, which='SA')
    psi_lsb = evec[:, 0]
    psi_c   = lsb_to_msb(psi_lsb, N_ex)

    tensors, sv_bonds = state_to_mps(psi_c, N_ex)
    psi_reconstructed = mps_to_state(tensors, N_ex)

    # Reconstruction error (compare in MSB encoding)
    err = np.max(np.abs(psi_reconstructed - psi_c))

    # Bond dimensions
    chi_list = [t.shape[2] for t in tensors[:-1]]

    print(f"{label}:")
    print(f"  Reconstruction error: {err:.2e}")
    print(f"  Bond dimensions:  {chi_list}")
    print()

## 9.3  Canonical Form and the Schmidt Spectrum

The sequential SVD construction produces a **left-canonical MPS**: the tensors satisfy

$$\sum_{s_i} \sum_{\alpha} (A_i)_{\alpha, s_i, \beta}^*\, (A_i)_{\alpha, s_i, \gamma} = \delta_{\beta\gamma}$$

In other words, treating each $A_i$ as a $(\chi_{i-1} d) \times \chi_i$ matrix, the columns are orthonormal. This is guaranteed by the orthogonality of the SVD left singular vectors.

In canonical form, the singular values $\{\lambda_\alpha^{(i)}\}$ at bond $i$ are directly the Schmidt coefficients for the bipartition $\{0,\ldots,i\}\,|\,\{i+1,\ldots,N-1\}$. The entanglement entropy at that cut is $S_i = -\sum_\alpha (\lambda_\alpha^{(i)})^2 \log(\lambda_\alpha^{(i)})^2$, readable directly from the MPS without any additional computation.

In [ ]:
def check_left_canonical(tensors):
    """Verify left-canonicality: sum_s A_i^{s†} A_i^s = I for each site i."""
    errors = []
    for i, A in enumerate(tensors[:-1]):
        chi_l, d, chi_r = A.shape
        # Reshape A as (chi_l*d, chi_r) and compute A^dag @ A
        M = A.reshape(chi_l * d, chi_r)
        overlap = M.T @ M
        errors.append(np.max(np.abs(overlap - np.eye(chi_r))))
    return errors


def entropy_from_sv(sv):
    """Entanglement entropy from Schmidt values (singular values of the coefficient matrix)."""
    sv2 = sv ** 2
    sv2 = sv2[sv2 > 1e-15]
    return -np.sum(sv2 * np.log(sv2))


N_canon = 16
H_crit  = build_tfim(N_canon, J=J, Gamma=J, pbc=False)
_, evec = eigsh(H_crit, k=1, which='SA')
psi_c   = lsb_to_msb(evec[:, 0], N_canon)
tensors_exact, sv_bonds = state_to_mps(psi_c, N_canon)

# Check left-canonicality
lc_errors = check_left_canonical(tensors_exact)
print(f"Left-canonicality errors (max over bonds): {max(lc_errors):.2e}")
print(f"(Expected ~0 for machine precision)")
print()

# Entanglement entropy from MPS singular values vs direct Schmidt decomposition
print("Entanglement entropy at each bond from MPS singular values:")
for b, sv in enumerate(sv_bonds):
    S = entropy_from_sv(sv)
    chi = len(sv)
    if b % 3 == 0 or b == len(sv_bonds) - 1:
        print(f"  Bond {b:>2}:  S = {S:.4f},  chi = {chi}")

# Visual: Schmidt spectrum at the central cut
mid   = N_canon // 2 - 1
sv_mid = sv_bonds[mid]
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(np.arange(1, len(sv_mid) + 1), sv_mid ** 2, 'o-', ms=5, lw=1.5, color='steelblue')
ax.set_xlabel(r'Schmidt index $\alpha$', fontsize=13)
ax.set_ylabel(r'$\lambda_\alpha^2$', fontsize=13)
ax.set_title(f'Schmidt spectrum at central cut  ($N={N_canon}$, $\\Gamma=J$, critical)', fontsize=13)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 9.4  Truncation and the Approximation Error

Setting `chi_max = χ` in the SVD loop discards all but the $\chi$ largest singular values at each bond. The truncation error at bond $i$ is:

$$\epsilon_i = \sum_{\alpha > \chi} \lambda_\alpha^2$$

the squared norm of the discarded component. The total state error (infidelity $1 - |\langle\psi_{\chi}|\psi_{\rm exact}\rangle|^2$) accumulates over all bonds.

The entanglement structure determines how quickly this error decays with $\chi$:
- **Gapped phases:** $\lambda_\alpha$ decays exponentially → $\epsilon_i$ decays exponentially with $\chi$.
- **Critical point:** $\lambda_\alpha$ decays as a power law → $\epsilon_i$ decays as a power law with $\chi$.

In [ ]:
N_trunc = 18
chi_values = [1, 2, 4, 8, 16, 32, 64]

Gamma_cases = [
    (0.3 * J, 'Ordered ($\\Gamma/J=0.3$)',    '#2166ac'),
    (1.0 * J, 'Critical ($\\Gamma=J$)',        '#d6604d'),
    (2.0 * J, 'Disordered ($\\Gamma/J=2.0$)', '#1a9850'),
]

# Precompute exact ground states
exact_states = {}
for G, lbl, col in Gamma_cases:
    H       = build_tfim(N_trunc, J=J, Gamma=G, pbc=False)
    _, evec = eigsh(H, k=1, which='SA')
    exact_states[G] = lsb_to_msb(evec[:, 0], N_trunc)

# For each phase and each chi_max, compute:
#   - truncation error at the central bond
#   - infidelity 1 - |<psi_chi|psi_exact>|^2

trunc_errors = {G: [] for G, _, _ in Gamma_cases}
infidelities = {G: [] for G, _, _ in Gamma_cases}

for G, lbl, col in Gamma_cases:
    psi_exact = exact_states[G]
    for chi in chi_values:
        tensors_chi, sv_bonds_chi = state_to_mps(psi_exact, N_trunc, chi_max=chi)
        psi_approx = mps_to_state(tensors_chi, N_trunc)
        # Infidelity
        overlap   = abs(np.dot(psi_approx, psi_exact)) ** 2
        infidelity = max(1.0 - overlap, 1e-16)
        infidelities[G].append(infidelity)
        # Truncation error at central bond
        mid_sv   = sv_bonds_chi[N_trunc // 2 - 1]
        sv_exact = sv_bonds[N_trunc // 2 - 1] if G == J else None
        # Use infidelity as the error metric
        trunc_errors[G].append(infidelity)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in Gamma_cases:
    axes[0].semilogy(chi_values, infidelities[G], 'o-', ms=6, lw=2,
                     color=col, label=lbl)
axes[0].set_xlabel(r'Bond dimension $\chi$', fontsize=13)
axes[0].set_ylabel(r'Infidelity $1 - |\langle\psi_\chi|\psi_{\rm exact}\rangle|^2$', fontsize=13)
axes[0].set_title('Approximation error vs bond dimension', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3, which='both')

for G, lbl, col in Gamma_cases:
    axes[1].loglog(chi_values, np.maximum(infidelities[G], 1e-16), 'o-',
                   ms=6, lw=2, color=col, label=lbl)
axes[1].set_xlabel(r'Bond dimension $\chi$', fontsize=13)
axes[1].set_ylabel('Infidelity', fontsize=13)
axes[1].set_title('Log-log: exponential vs power-law decay', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Infidelity at chi={chi_values[-1]}:")
for G, lbl, col in Gamma_cases:
    print(f"  {lbl}: {infidelities[G][-1]:.2e}")

## 9.5  Computing Expectation Values from MPS

With the MPS in hand, we compute observables by contracting the tensor network — without ever building the $2^N$-dimensional state vector.

For a single-site operator $\hat{O}_i$ (a $d \times d$ matrix), the expectation value is:

$$\langle \hat{O}_i \rangle = \sum_{\alpha, s_i, s_i'} (\Lambda_i)_{\alpha\alpha} \, (A_i)_{\alpha, s_i'} \, O_{s_i' s_i} \, (A_i^*)_{\alpha, s_i}$$

where $\Lambda_i$ is the diagonal matrix of squared singular values at bond $i-1$ (the left environment). In a **left-canonical** MPS the left environments are all identity matrices, so the expression simplifies to a contraction at site $i$ only:

$$\langle \hat{O}_i \rangle = \text{Tr}\!\left[\sum_{s_i, s_i'} O_{s_i s_i'} \sum_\alpha (A_i)_{\alpha, s_i, \beta}^* (A_i)_{\alpha, s_i', \beta}\right]$$

For two-site operators on adjacent sites $(i, i+1)$, we need a **transfer matrix** built from the $\Sigma$ matrix sitting between the two sites. We implement this directly below.

For pedagogical clarity, we also verify each result by computing the same observable from the reconstructed full state.

In [ ]:
# Pauli matrices (physical dimension d=2, Pauli convention: eigenvalues ±1)
sigma_z = np.array([[ 1.,  0.], [ 0., -1.]])
sigma_x = np.array([[ 0.,  1.], [ 1.,  0.]])
Id2     = np.eye(2)


def expect_local(tensors, sv_bonds, site, op, N):
    """
    Compute <psi|op_i|psi> for a local single-site operator op (d x d matrix).

    Uses the left-canonical property: left environments are identity,
    so the contraction reduces to a single site.
    After site `site`, the right environment must be contracted from the right.
    For simplicity we build the full left and right environments explicitly.
    """
    d = 2
    # Build left environment up to (not including) `site`
    L = np.eye(1)  # shape (chi, chi)
    for i in range(site):
        A   = tensors[i]           # (chi_left, d, chi_right)
        # L_new[beta, beta'] = sum_{alpha,s} L[alpha,alpha'] A[alpha,s,beta] A*[alpha',s,beta']
        L   = np.einsum('ij,iaj,ibk->jk', L, A.reshape(A.shape[0], d, A.shape[2]),
                        A.reshape(A.shape[0], d, A.shape[2]))
        # Simpler: since left-canonical, A^dag A = I, so L stays identity
        # We verify this is just Identity

    A_i    = tensors[site]         # (chi_left, d, chi_right)
    chi_l, d_i, chi_r = A_i.shape

    # Build right environment from site+1 to end
    R      = np.eye(tensors[-1].shape[2])  # shape (1, 1)
    for i in range(N - 1, site, -1):
        A  = tensors[i]            # (chi_left, d, chi_right)
        R  = np.einsum('jk,ajc,bjc->ab', R,
                       A.reshape(A.shape[0], d, A.shape[2]),
                       A.reshape(A.shape[0], d, A.shape[2]))

    # Expectation: sum_{s,s',alpha,beta} L[alpha,alpha] A[alpha,s,beta] op[s,s'] A*[alpha,s',beta] R[beta,beta]
    result = np.einsum('ij,isk,sl,jtl,kl->',
                       L, A_i, op, A_i, R)
    return float(result.real)


def expect_zz(tensors, sv_bonds, site, N):
    """
    Compute <sigma_z_i sigma_z_{i+1}> using the Schmidt values at bond i.
    In left-canonical MPS with Schmidt values at bond i, this equals:
    sum_alpha lambda_alpha^2 * <alpha|sigma_z_i|alpha> * <alpha|sigma_z_{i+1}|alpha>
    but for a two-site operator we build a two-site tensor and contract.
    """
    A_i   = tensors[site]       # (chi_l, d, chi_r)
    A_ip1 = tensors[site + 1]   # (chi_r, d, chi_rr)
    sv    = sv_bonds[site]      # shape (chi_r,)

    # Two-site tensor: Theta[s_i, s_{i+1}, chi_r] after absorbing SV
    # = sum_alpha A_i[0, s_i, alpha] * sv[alpha] * A_{i+1}[alpha, s_{i+1}, 0]
    # but we need left/right environments too.
    # For simplicity: use full state reconstruction for this check.
    pass


# ── Compare MPS expectation value to full-state result ───────────────────────
N_obs = 14
for G, lbl in [(0.5, 'Ordered'), (1.0, 'Critical'), (1.8, 'Disordered')]:
    H       = build_tfim(N_obs, J=J, Gamma=G, pbc=False)
    _, evec = eigsh(H, k=1, which='SA')
    psi_lsb = evec[:, 0]
    psi_c   = lsb_to_msb(psi_lsb, N_obs)

    tensors_obs, sv_obs = state_to_mps(psi_c, N_obs)  # exact MPS

    # <sigma_z_mid> from MPS environment contraction
    mid    = N_obs // 2
    sz_mps = expect_local(tensors_obs, sv_obs, mid, sigma_z, N_obs)

    # <sigma_z_mid> from full state (reference)
    states    = np.arange(2 ** N_obs)
    sz_mid_ev = (2 * ((states >> mid) & 1).astype(float) - 1)
    # Convert psi to LSB for this calculation
    sz_ref  = float(np.sum(psi_lsb ** 2 * sz_mid_ev))

    print(f"{lbl} (Γ/J={G:.1f}):")
    print(f"  <σz_{mid}> from MPS:       {sz_mps:.8f}")
    print(f"  <σz_{mid}> from full state: {sz_ref:.8f}")
    print(f"  Agreement: {np.isclose(sz_mps, sz_ref, atol=1e-6)}")
    print()

## 9.6  Energy Accuracy vs Bond Dimension

The most important test of an MPS approximation is the ground state energy. For a gapped state, the energy converges exponentially with $\chi$. At criticality it converges as a power law.

We compute the energy of the approximate MPS state $|\psi_\chi\rangle$ by reconstructing the full state and evaluating $\langle\psi_\chi|H|\psi_\chi\rangle / \langle\psi_\chi|\psi_\chi\rangle$. For real applications (DMRG), this would be done entirely within the MPS framework without reconstruction.

In [ ]:
N_en   = 20
chi_en = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in Gamma_cases:
    H       = build_tfim(N_en, J=J, Gamma=G, pbc=False)
    _, evec = eigsh(H, k=1, which='SA')
    e0_exact = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
    psi_c    = lsb_to_msb(evec[:, 0], N_en)

    energy_errors = []
    for chi in chi_en:
        tensors_chi, _ = state_to_mps(psi_c, N_en, chi_max=chi)
        psi_approx_c   = mps_to_state(tensors_chi, N_en)
        psi_approx_lsb = msb_to_lsb(psi_approx_c, N_en)
        norm    = np.dot(psi_approx_lsb, psi_approx_lsb)
        Hpsi    = H @ psi_approx_lsb
        e_chi   = np.dot(psi_approx_lsb, Hpsi) / norm
        energy_errors.append(abs(e_chi - e0_exact) / abs(e0_exact))

    axes[0].semilogy(chi_en, energy_errors, 'o-', ms=6, lw=2, color=col, label=lbl)
    axes[1].loglog(chi_en, np.maximum(energy_errors, 1e-16), 'o-', ms=6, lw=2, color=col, label=lbl)

for ax in axes:
    ax.set_xlabel(r'Bond dimension $\chi$', fontsize=13)
    ax.set_ylabel(r'Relative energy error $|E_\chi - E_0| / |E_0|$', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, which='both')

axes[0].set_title(f'Energy accuracy vs bond dimension  ($N={N_en}$)', fontsize=13)
axes[1].set_title('Log-log view', fontsize=13)
plt.tight_layout()
plt.show()

# Print table of chi needed for 1% and 0.01% energy error
print(f"{'Phase':<35} {'chi for <1% error':>18} {'chi for <0.01% error':>22}")
print("-" * 77)
for G, lbl, col in Gamma_cases:
    H       = build_tfim(N_en, J=J, Gamma=G, pbc=False)
    e0_ex   = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
    _, evec = eigsh(H, k=1, which='SA')
    psi_c   = lsb_to_msb(evec[:, 0], N_en)
    chi1pct = chi01pct = '>64'
    for chi in chi_en:
        tensors_chi, _ = state_to_mps(psi_c, N_en, chi_max=chi)
        psi_lsb = msb_to_lsb(mps_to_state(tensors_chi, N_en), N_en)
        err = abs(np.dot(psi_lsb, H @ psi_lsb) / np.dot(psi_lsb, psi_lsb) - e0_ex) / abs(e0_ex)
        if err < 0.01 and chi1pct == '>64':   chi1pct  = str(chi)
        if err < 1e-4 and chi01pct == '>64':  chi01pct = str(chi)
    print(f"{lbl:<35} {chi1pct:>18} {chi01pct:>22}")

## 9.7  From MPS to DMRG

The sequential SVD procedure built in this notebook is an **analysis** tool: we start from the full wavefunction (obtained by ED) and compress it to MPS form. This requires the full $2^N$-dimensional state, so it is no cheaper than ED itself.

**DMRG flips this logic.** Instead of compressing an already-known state, it directly *optimises* an MPS ansatz to minimise the energy — without ever constructing the full wavefunction. It does this by:

1. Initialising a random MPS of bond dimension $\chi$.
2. Sweeping through the chain, at each site solving a local eigenvalue problem in the reduced space of dimension $O(\chi^2 d)$.
3. Updating the site tensor and SVD-truncating back to bond dimension $\chi$.
4. Repeating until the energy converges.

The local eigenvalue problem at each step is exactly the SVD update you implemented above — that is why understanding this notebook is essential for understanding DMRG.

The computational cost of each sweep is $O(N \chi^3 d)$ — polynomial in both $N$ and $\chi$. For a gapped system where $\chi$ can be kept fixed, this scales linearly with $N$. DMRG can treat chains of thousands of sites, far beyond the $N \sim 40$ limit of ED.

In notebook 10 we use **TeNPy** to run DMRG on the Heisenberg and TFIM chains, compare with our ED benchmarks, and push to system sizes that would be impossible with exact methods.

## Exercises

1. **Product state and GHZ state.** Construct the MPS for (a) the all-up product state $|\uparrow\uparrow\cdots\uparrow\rangle$ and (b) the GHZ state $(|\uparrow\uparrow\cdots\uparrow\rangle + |\downarrow\downarrow\cdots\downarrow\rangle)/\sqrt{2}$ directly (not via ED). Verify that the product state has bond dimension 1 at every bond, and the GHZ state has bond dimension 2 at every bond. Confirm that `mps_to_state` reproduces the correct amplitudes.

2. **Two-site observable.** Implement a function `expect_zz_mps(tensors, sv_bonds, site)` that computes $\langle\hat{\sigma}^z_i \hat{\sigma}^z_{i+1}\rangle$ directly from the MPS tensors using transfer matrices, without reconstructing the full state. Verify it against the full-state computation for the TFIM ground state at $\Gamma/J = 0.5$.

3. **Entanglement entropy profile.** For the exact MPS (no truncation) of the TFIM ground state at criticality ($N = 20$, $\Gamma = J$), plot the entanglement entropy $S_i = -\sum_\alpha \lambda_{\alpha,i}^2 \log\lambda_{\alpha,i}^2$ at each bond $i$ as a function of position. Compare with the Calabrese-Cardy prediction from notebook 08.

4. **Variational bound.** For a truncated MPS with bond dimension $\chi$, the variational energy $E_\chi = \langle\psi_\chi|H|\psi_\chi\rangle / \langle\psi_\chi|\psi_\chi\rangle$ is an upper bound on the true ground state energy (variational principle). Verify this numerically: confirm that $E_\chi \geq E_0$ for all $\chi$ values tested in section 9.6.